# Exemplos — Configuração OpenSearch + vLLM para Produção
## Referência Técnica — Aula 9

**MBA: RAG & CAG Aplicados a Direito e Segurança Pública**

---

Este notebook contém exemplos de configuração de produção para os dois principais componentes da stack:
- OpenSearch com buscas híbridas e configuração de segurança
- vLLM com modelos open-source

Use como referência ao configurar o ambiente institucional.

## Exemplo 1: Configuração OpenSearch — docker-compose.yml para Produção

```yaml
# docker-compose-opensearch-prod.yml
# Cluster OpenSearch de 3 nós para produção em segurança pública

version: '3.8'
services:
  opensearch-node1:
    image: opensearchproject/opensearch:2.18.0
    container_name: opensearch-node1
    environment:
      - cluster.name=juridico-cluster
      - node.name=opensearch-node1
      - discovery.seed_hosts=opensearch-node1,opensearch-node2,opensearch-node3
      - cluster.initial_cluster_manager_nodes=opensearch-node1
      - bootstrap.memory_lock=true
      - "OPENSEARCH_JAVA_OPTS=-Xms4g -Xmx4g"
      - OPENSEARCH_INITIAL_ADMIN_PASSWORD=${OPENSEARCH_ADMIN_PWD}
    ulimits:
      memlock: { soft: -1, hard: -1 }
      nofile: { soft: 65536, hard: 65536 }
    volumes:
      - opensearch-data1:/usr/share/opensearch/data
      - ./config/opensearch.yml:/usr/share/opensearch/config/opensearch.yml
    ports:
      - 9200:9200
    networks: [opensearch-net]

  opensearch-dashboards:
    image: opensearchproject/opensearch-dashboards:2.18.0
    ports: ["5601:5601"]
    environment:
      OPENSEARCH_HOSTS: '["https://opensearch-node1:9200"]'
    networks: [opensearch-net]

volumes:
  opensearch-data1:

networks:
  opensearch-net:
```

## Exemplo 2: Configurar Pipeline de Busca Híbrida no OpenSearch

O OpenSearch requer a criação de um **search pipeline** com normalization processor para que a busca híbrida funcione corretamente.

In [ ]:
# Exemplo 2.1 — Criar pipeline de normalização para busca híbrida
import requests

OPENSEARCH_URL = "http://localhost:9200"

# Criar pipeline de busca híbrida com normalização min-max
pipeline_config = {
    "description": "Pipeline de busca híbrida para documentos jurídicos",
    "phase_results_processors": [
        {
            "normalization-processor": {
                "normalization": {
                    "technique": "min_max"  # Normaliza scores BM25 e kNN para [0,1]
                },
                "combination": {
                    "technique": "arithmetic_mean",
                    "parameters": {
                        "weights": [0.3, 0.7]  # 30% BM25, 70% kNN
                    }
                }
            }
        }
    ]
}

resp = requests.put(
    f"{OPENSEARCH_URL}/_search/pipeline/juridico-hybrid-pipeline",
    json=pipeline_config,
    headers={"Content-Type": "application/json"}
)

if resp.status_code in [200, 201]:
    print("✅ Pipeline de busca híbrida criado com sucesso")
else:
    print(f"Status: {resp.status_code}")
    print(resp.text[:300])

In [ ]:
# Exemplo 2.2 — Usar pipeline na busca híbrida
# Note o parâmetro search_pipeline na URL

query_hibrida_com_pipeline = {
    "query": {
        "hybrid": {
            "queries": [
                {
                    "match": {
                        "conteudo": "colaboração premiada requisitos"
                    }
                },
                {
                    "knn": {
                        "embedding": {
                            "vector": [0.1] * 1024,  # Substitua pelo embedding real
                            "k": 10
                        }
                    }
                }
            ]
        }
    },
    "size": 5
}

# Executar com pipeline
resp = requests.post(
    f"{OPENSEARCH_URL}/juridico-busca-hibrida/_search"
    f"?search_pipeline=juridico-hybrid-pipeline",
    json=query_hibrida_com_pipeline,
    headers={"Content-Type": "application/json"}
)

print(f"Status: {resp.status_code}")
if resp.status_code == 200:
    dados = resp.json()
    print(f"Resultados: {dados['hits']['total']['value']}")
    for hit in dados['hits']['hits'][:3]:
        print(f"  Score: {hit['_score']:.4f} | {hit['_source'].get('titulo', 'N/A')[:60]}")

## Exemplo 3: Configurar vLLM para Produção

```bash
# Iniciar vLLM com Llama-3.1-8B-Instruct
# Requer GPU com pelo menos 16GB VRAM

python -m vllm.entrypoints.openai.api_server \
    --model meta-llama/Llama-3.1-8B-Instruct \
    --host 0.0.0.0 \
    --port 8000 \
    --max-model-len 8192 \
    --dtype bfloat16 \
    --tensor-parallel-size 1 \
    --gpu-memory-utilization 0.90 \
    --api-key "token-institucional-seguro" \
    --served-model-name "llama-juridico"

# Para modelos maiores (70B) com 2 GPUs:
python -m vllm.entrypoints.openai.api_server \
    --model meta-llama/Llama-3.1-70B-Instruct \
    --tensor-parallel-size 2 \
    --max-model-len 32768 \
    --dtype bfloat16
```

In [ ]:
# Exemplo 3.1 — Listar modelos disponíveis no servidor vLLM
import requests

VLLM_URL = "http://localhost:8000"
VLLM_KEY = "token-institucional-seguro"

try:
    resp = requests.get(
        f"{VLLM_URL}/v1/models",
        headers={"Authorization": f"Bearer {VLLM_KEY}"}
    )
    if resp.status_code == 200:
        modelos = resp.json()
        print("✅ Modelos disponíveis no vLLM:")
        for m in modelos.get('data', []):
            print(f"  • {m['id']}")
    else:
        print(f"Status: {resp.status_code} — vLLM não acessível")
except Exception as e:
    print(f"vLLM não disponível: {e}")
    print("Configure VLLM_URL com o endereço do servidor institucional")

In [ ]:
# Exemplo 3.2 — Configuração completa do LightRAG para produção
# Template pronto para uso em ambiente institucional

CONFIGURACAO_PRODUCAO = """
# lightrag_config_producao.py
# Configuração de produção — Segurança Pública

import os
from lightrag import LightRAG, QueryParam
from FlagEmbedding import FlagModel, FlagReranker

# === CONFIGURAÇÕES DO AMBIENTE ===
OPENSEARCH_HOST = os.environ["OPENSEARCH_HOST"]       # Ex: opensearch.intranet.gov.br
OPENSEARCH_PORT = int(os.environ.get("OPENSEARCH_PORT", "9200"))
OPENSEARCH_USER = os.environ["OPENSEARCH_USER"]       # Usuário RBAC
OPENSEARCH_PASS = os.environ["OPENSEARCH_PASS"]       # Senha do usuário

VLLM_URL   = os.environ["VLLM_URL"]                   # Ex: http://vllm.intranet.gov.br:8000/v1
VLLM_MODEL = os.environ.get("VLLM_MODEL", "llama-juridico")
VLLM_KEY   = os.environ["VLLM_API_KEY"]

# === MODELOS LOCAIS ===
embedding_model = FlagModel('BAAI/bge-m3', use_fp16=True)
reranker_model  = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)

# === LIGHTRAG ===
rag = LightRAG(
    llm_model_func=...,          # Função async configurada
    embedding_func=...,          # Função async de embedding
    embedding_dim=1024,
    llm_model_max_token_size=32768,
    llm_model_max_async=8,
    embedding_batch_num=32,
    chunk_token_size=1200,
    chunk_overlap_token_size=150,
    kv_storage="OpensearchKVStorage",
    vector_storage="OpensearchVectorDBStorage",
    graph_storage="OpensearchGraphStorage",
    doc_status_storage="JsonDocStatusStorage",
    working_dir="/data/lightrag/juridico",
    addon_params={
        "opensearch_config": {
            "host": OPENSEARCH_HOST,
            "port": OPENSEARCH_PORT,
            "use_ssl": True,
            "verify_certs": True,
            "http_auth": (OPENSEARCH_USER, OPENSEARCH_PASS),
            "index_prefix": "juridico-prod"
        },
        "language": "Portuguese",
        "entity_extract_max_gleaning": 2,   # Tentativas de extração por chunk
    }
)
"""

print("Template de configuração de produção:")
print(CONFIGURACAO_PRODUCAO)

## Exemplo 4: Monitoramento do Grafo com OpenSearch Dashboards

Após indexar o corpus, use o OpenSearch Dashboards (porta 5601) para visualizar:

1. **Index Management:** ver tamanho e saúde dos índices LightRAG
2. **Dev Tools:** executar queries Elasticsearch-compatíveis diretamente
3. **Discover:** explorar documentos e entidades indexadas

Queries úteis no Dev Tools:

```json
// Contar entidades por tipo
GET juridico-prod-entities/_search
{
  "aggs": {
    "por_tipo": {
      "terms": { "field": "entity_type" }
    }
  },
  "size": 0
}

// Buscar entidades com maior grau de conexão
GET juridico-prod-entities/_search
{
  "sort": [{"degree": {"order": "desc"}}],
  "size": 10,
  "_source": ["entity_name", "entity_type", "degree"]
}
```